To measure code execution or round-trip time quickly in Python, use the built-in time module. 
The script below uses time.perf_counter() to measure how long a dummy task or a mock network call takes down to the microsecond.

In [1]:
import time 
import random 

start_time = time.perf_counter()
# Simulate some work with a random sleep
time.sleep(random.uniform(0.1, 0.5))
end_time = time.perf_counter()
latency = (end_time - start_time) * 1000  # Convert to milliseconds
print(f"Latency: {latency:.2f} ms")


Latency: 147.67 ms


In [2]:
#Another example of measuring latency

import time
import random

# Start the high-resolution performance timer
start_time = time.perf_counter()

# Dummy task: simulate a network delay or calculation
fake_latency = random.uniform(0.05, 0.2)  # 50ms to 200ms
time.sleep(fake_latency)

# Stop the timer
end_time = time.perf_counter()

# Calculate elapsed time in milliseconds
latency_ms = (end_time - start_time) * 1000

print(f"Measured Latency: {latency_ms:.2f} ms")


Measured Latency: 91.99 ms


How It Works

    Timer Start: time.perf_counter() begins tracking time with high accuracy.
    Simulation: time.sleep() mimics waiting for a server reply or data process.
    Calculation: Subtract the start time from the end time and multiply by 1,000 to convert seconds into milliseconds.

In [3]:
# Using timeit for more accurate latency measurement
import timeit

def dummy_function():
    time.sleep(0.1)

# Measure the execution time of the dummy function
latency = timeit.timeit(dummy_function, number=1) * 1000  # Convert to milliseconds
print(f"Latency: {latency:.2f} ms")

Latency: 103.02 ms


In [4]:
import time

# Using time.time() - Vulnerable to system clock adjustments
start = time.time()
time.sleep(0.1)
# If the system clock updates here, 'end' could be smaller than 'start'!
end = time.time() 
print(f"Time: {(end - start) * 1000} ms")

# Using time.perf_counter() - Always safe and highly precise
start_perf = time.perf_counter()
time.sleep(0.1)
end_perf = time.perf_counter()
print(f"Perf: {(end_perf - start_perf) * 1000} ms")


Time: 105.1948070526123 ms
Perf: 104.66154100140557 ms


### Why time.perf_counter() is Better?

System Clock Changes: time.time() tracks the real-world calendar time. If your computer automatically syncs with an internet time server during your test, the clock can jump forward or backward, breaking your measurement.

Monotonic Clock: time.perf_counter() uses a monotonic clock. This clock only goes forward and cannot be adjusted by the system, ensuring steady results.

Higher Resolution: time.perf_counter() utilizes the highest resolution timer available on your specific hardware, making it accurate down to microseconds.

## Illustration

Here are two ways to level up your latency demonstration: using the built-in timeit module to benchmark a real function, and using the requests library to measure actual network response times.

1. Benchmark a Real Function (Using timeit)
The timeit module isolates code execution and runs it thousands of times to get an accurate average speed, automatically eliminating background system spikes.

In [5]:
import timeit

# Define a real function to test
def list_comprehension_test():
    return [x * 2 for x in range(10000)]

# Run the function 1,000 times and get the total time
total_time = timeit.timeit(list_comprehension_test, number=1000)

# Calculate average latency per execution in milliseconds
avg_latency_ms = (total_time / 1000) * 1000

print(f"Average Execution Latency: {avg_latency_ms:.4f} ms")


Average Execution Latency: 0.1607 ms


2. Measure Real Network Latency (Using requests)

This script hits a live API and measures HTTP round-trip latency using the library's built-in .elapsed property, which internally relies on a high-resolution timer.

In [6]:
import requests

try:
    # Send a request to a fast, public API
    response = requests.get("https://github.com", timeout=5)
    
    # Get latency as a timedelta object and convert to milliseconds
    network_latency_ms = response.elapsed.total_seconds() * 1000
    
    print(f"Status Code: {response.status_code}")
    print(f"Network Round-Trip Latency: {network_latency_ms:.2f} ms")

except requests.exceptions.RequestException as e:
    print(f"Connection failed: {e}")


Status Code: 200
Network Round-Trip Latency: 105.23 ms
